## Week 2 - Japanese Hiragana Tutor
This notebook demonstrates a simple AI tutor for learning Japanese hiragana. It helps users understand each character by providing pronunciation, and a visual reference in one place.

In [ ]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import base64
from io import BytesIO
from PIL import Image
import random
from datetime import datetime

In [ ]:
# Init

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

IMG_MODEL = 'gpt-image-1-mini'
TTS_MODEL = 'gpt-4o-mini-tts'
MODEL = "gpt-4.1-mini"
openai = OpenAI()
DB = "hiragana.db"

HIRAGANA_BASE = [
    ("あ", "a"), ("い", "i"), ("う", "u"), ("え", "e"), ("お", "o"),
    ("か", "ka"), ("き", "ki"), ("く", "ku"), ("け", "ke"), ("こ", "ko"),
    ("さ", "sa"), ("し", "shi"), ("す", "su"), ("せ", "se"), ("そ", "so"),
    ("た", "ta"), ("ち", "chi"), ("つ", "tsu"), ("て", "te"), ("と", "to"),
    ("な", "na"), ("に", "ni"), ("ぬ", "nu"), ("ね", "ne"), ("の", "no"),
    ("は", "ha"), ("ひ", "hi"), ("ふ", "fu"), ("へ", "he"), ("ほ", "ho"),
    ("ま", "ma"), ("み", "mi"), ("む", "mu"), ("め", "me"), ("も", "mo"),
    ("や", "ya"), ("ゆ", "yu"), ("よ", "yo"), ("ら", "ra"), ("り", "ri"),
    ("る", "ru"), ("れ", "re"), ("ろ", "ro"), ("わ", "wa"), ("を", "wo"),
    ("ん", "n")
]

system_message = """
You are a enthusiastic and encouraging hiragana tutor helping a student learn to read and write hiragana characters.

When the student asks for a new character or practice, get one for them.
When the student gives an answer for a character, check if it is correct and record the attempt.
When the student asks about their progress, look it up and summarize it clearly.

Give short, encouraging answers, no more than 2 sentences.
If the student answers correctly, briefly confirm it. If incorrect, tell them the correct romaji.
Always be accurate. If you don't know the answer, say so.
If you don't know the answer, say so.
"""

current_char_state = {
    "character": None,
    "romaji": None,
    "image": None,
    "audio": None
}

In [ ]:
# Data Init

def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        # Create the table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS progress (
            character TEXT PRIMARY KEY,
            romaji TEXT NOT NULL,
            attempts INTEGER DEFAULT 0,
            correct INTEGER DEFAULT 0,
            last_seen TIMESTAMP
        )
        """)

        # Verify creation
        count = cursor.execute("SELECT COUNT(*) FROM progress").fetchone()[0]

        # If the count is 0 means the table exists but is empty. So its good to insert
        if count == 0:
            cursor.executemany(
                "INSERT INTO progress (character, romaji) VALUES (?, ?)",
                HIRAGANA_BASE
            )

init_db()

In [ ]:
# Tools

def get_next_character():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        row = cursor.execute(
            "SELECT character, romaji FROM progress ORDER BY RANDOM() LIMIT 1"
        ).fetchone()

    return row[0], row[1]

def record_attempt(character, is_correct):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        row = conn.execute(
            "SELECT attempts, correct FROM progress WHERE character = ?",
            (character,)
        ).fetchone()

        attempts, correct = row
        attempts += 1
        if is_correct:
            correct += 1

        conn.execute(
            "UPDATE progress SET attempts = ?, correct = ?, last_seen = ? WHERE character = ?",
            (attempts, correct, datetime.now(), character)
        )

def check_answer(correct_romaji, user_input):
    return user_input.strip().lower() == correct_romaji.strip().lower()

def get_progress_summary():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        rows = conn.execute(
            "SELECT character, romaji, attempts, correct FROM progress"
        ).fetchall()

        return rows


# get_next_character()

In [ ]:
# Tool Definations

get_next_character_function = {
    "name": "get_next_character",
    "description": "Get a random hiragana character and its romaji from the database for the student to practice.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

record_attempt_function = {
    "name": "record_attempt",
    "description": "Record whether the student answered a hiragana character correctly, updating their attempt count and correct count in the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "character": {
                "type": "string",
                "description": "The hiragana character the student was practicing",
            },
            "is_correct": {
                "type": "boolean",
                "description": "Whether the student's answer was correct",
            },
        },
        "required": ["character", "is_correct"],
        "additionalProperties": False
    }
}

check_answer_function = {
    "name": "check_answer",
    "description": "Check if the student's typed romaji answer matches the correct romaji for a hiragana character.",
    "parameters": {
        "type": "object",
        "properties": {
            "correct_romaji": {
                "type": "string",
                "description": "The correct romaji spelling for the character",
            },
            "user_input": {
                "type": "string",
                "description": "The romaji the student typed as their answer",
            },
        },
        "required": ["correct_romaji", "user_input"],
        "additionalProperties": False
    }
}

get_progress_summary_function = {
    "name": "get_progress_summary",
    "description": "Get the student's full practice history, including every character, its romaji, attempts, and correct count.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": get_next_character_function},
    {"type": "function", "function": record_attempt_function},
    {"type": "function", "function": check_answer_function},
    {"type": "function", "function": get_progress_summary_function},
]
tools

In [ ]:
def generate_character_image(character):
    image_prompt = f"""
        Generate an image containing only the completed Japanese hiragana character "{character}".
        Use black ink on a white background.
        Center the character and make it large.
        No stroke order, no numbers, no arrows, no labels, no text, no grid, no additional symbols.
    """

    image_response = openai.images.generate(
        model=IMG_MODEL,
        prompt=image_prompt,
        size="1024x1024",
        n=1,
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

def generate_tts(message):
    response = openai.audio.speech.create(
        model=TTS_MODEL,
        voice="fable",
        input=message
    )
    return response.content

# image = generate_character_image("ち")
# display(image)

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_next_character":
            character, romaji = get_next_character()
            image = generate_character_image(character)
            audio = generate_tts(character)

            current_char_state["character"] = character
            current_char_state["romaji"] = romaji
            current_char_state["image"] = image
            current_char_state["audio"] = audio

            result = {"character": character, "romaji": romaji}

        elif tool_call.function.name == "record_attempt":
            character = arguments.get("character")
            is_correct = arguments.get("is_correct")
            record_attempt(character, is_correct)
            result = {"status": "recorded"}

        elif tool_call.function.name == "check_answer":
            correct_romaji = arguments.get("correct_romaji")
            user_input = arguments.get("user_input")
            is_correct = check_answer(correct_romaji, user_input)
            result = {"is_correct": is_correct}

        elif tool_call.function.name == "get_progress_summary":
            rows = get_progress_summary()
            result = {"progress": rows}

        responses.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return responses
        

def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    return history, current_char_state["image"], current_char_state["audio"], ""

In [ ]:
# Callbacks (along with the chat() function above)
def put_message_in_chatbot(message, history):
    return history + [{"role": "user", "content": message}], ""

# UI Def
with gr.Blocks() as ui:
    gr.Markdown("# Hiragana Tutor")

    with gr.Row():
        image_output = gr.Image(label="Character", height=300, interactive=False)
        audio_output = gr.Audio(label="Pronunciation")

    chatbot = gr.Chatbot(label="Chat", type="messages",)
    msg_input = gr.Textbox(label="Enter Your Message")
    send_btn = gr.Button("Send")

    send_btn.click(
        put_message_in_chatbot,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, msg_input]
    ).then(
        chat,
        inputs=[chatbot],
        outputs=[chatbot, image_output, audio_output, msg_input]
    )

ui.launch(inbrowser=True, auth=("admin", "strongpassword"))

## Future Updates
- Add a difficulty column - Starts with a base 1 and increase or decrease it when a student gets an answer wrong or correct
- Search for a model which is closer to a Japanese voice for correct pronounciation
- Stroke order for each hiragana character??